In [1]:
import subprocess
import sys

# Install uv for faster package management
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv', '-q'], check=True)

try:
    from autogluon.tabular import TabularPredictor
except ImportError:
    subprocess.run(['uv', 'pip', 'install', 'autogluon', '--system', '-q'], check=True)
    from autogluon.tabular import TabularPredictor

import pandas as pd
import numpy as np
import glob
import re


def find_data_files() -> tuple[str, str, str]:
    """Find train.csv, test.csv, and sample_submission.csv files in /kaggle/input."""
    files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    train_file = next(f for f in files if re.search(r'train\.csv$', f))
    test_file = next(f for f in files if re.search(r'test\.csv$', f))
    sample_file = next(f for f in files if re.search(r'sample.*submission\.csv$', f, re.IGNORECASE))
    return train_file, test_file, sample_file


def load_data() -> tuple[pd.DataFrame, pd.DataFrame, str, str]:
    """Load train and test datasets. Target is the column in train but not in test. Id is the shared column between test and sample_submission."""
    train_file, test_file, sample_file = find_data_files()
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    sample_df = pd.read_csv(sample_file, nrows=0)
    target = (set(train_df.columns) - set(test_df.columns)).pop()
    id_col = (set(test_df.columns) & set(sample_df.columns)).pop()
    return train_df, test_df, target, id_col


def create_submission(test_df: pd.DataFrame, predictions: np.ndarray, target: str, id_col: str) -> pd.DataFrame:
    """Create submission dataframe."""
    return pd.DataFrame({
        id_col: test_df[id_col],
        target: predictions
    })


def main():
    train_df, test_df, target, id_col = load_data()

    X_train = train_df.drop(columns=[id_col])
    X_test = test_df.drop(columns=[id_col])

    predictor = TabularPredictor(
        label=target,
        eval_metric='root_mean_squared_error'
    ).fit(
        X_train,
        time_limit=1200,
        presets='best_quality',
    )

    predictions = predictor.predict(X_test)

    submission = create_submission(test_df, predictions, target, id_col)
    submission.to_csv('submission.csv', index=False)

    print(f"\nLeaderboard:")
    print(predictor.leaderboard())
    print(f"\nSubmission shape: {submission.shape}")
    print(submission.head())


if __name__ == '__main__':
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 80.1 MB/s eta 0:00:00


No path specified. Models will be saved in: "AutogluonModels/ag-20260113_044701"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Sep 27 10:16:09 UTC 2025
CPU Count:          4
Pytorch Version:    2.8.0+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 14.74/14.74 GB | GPU 1: 14.74/14.74 GB
Total GPU Memory:   Free: 29.48 GB, Allocated: 0.00 GB, Total: 29.48 GB
GPU Count:          2
Memory Avail:       29.83 GB / 31.35 GB (95.1%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enable


Leaderboard:
                     model     score_val              eval_metric  \
0      WeightedEnsemble_L3 -23603.657875  root_mean_squared_error   
1      WeightedEnsemble_L2 -23807.992729  root_mean_squared_error   
2           XGBoost_BAG_L2 -24495.419763  root_mean_squared_error   
3        LightGBMXT_BAG_L1 -24869.397020  root_mean_squared_error   
4     CatBoost_r177_BAG_L1 -25055.626371  root_mean_squared_error   
5     CatBoost_r177_BAG_L2 -25506.497661  root_mean_squared_error   
6          CatBoost_BAG_L2 -25571.744046  root_mean_squared_error   
7        LightGBMXT_BAG_L2 -25721.717860  root_mean_squared_error   
8   RandomForestMSE_BAG_L2 -25728.573612  root_mean_squared_error   
9     ExtraTreesMSE_BAG_L2 -25786.928017  root_mean_squared_error   
10         CatBoost_BAG_L1 -25798.951981  root_mean_squared_error   
11         LightGBM_BAG_L2 -26102.929829  root_mean_squared_error   
12   NeuralNetTorch_BAG_L2 -26323.516766  root_mean_squared_error   
13   NeuralNetTorch_